# Faruq-v3 — Multi-Model Complementarity Audit (seed42)

Validation-only post-training audit untuk CMC0, STB1, AF2, IGEM1, SAF1, dan ACMC1. Tidak ada training dan test tidak diekstrak. Setiap checkpoint dibuka dari branch asalnya, diekspor sebagai object-event JSON netral, lalu dibandingkan pairwise.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, os, shutil, subprocess, sys, tarfile
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
AUDIT_BRANCH = 'agent/multimodel-complementarity-audit'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', AUDIT_BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
EXPORTER = Path('/content/export_validation_object_events.py')
shutil.copy2(REPO / 'scripts/export_validation_object_events.py', EXPORTER)
print('AUDIT COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

BUNDLE_REL = 'bundles/faruq-development-v3-grouped.tar'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(BUNDLE_REL,))
ARCHIVE = require_project_artifact(PROJECT_ROOT, BUNDLE_REL)
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'

SPECS = {
    'CMC0': ('agent/stb-capacity-causal-control', 'experiments/faruq-v3-stb-capacity-control-v1/CMC0_seed42/weights/best.pt'),
    'STB1': ('agent/gds-stb-classification-screening', 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/STB1/STB1_seed42/weights/best.pt'),
    'AF2': ('agent/lfdet-afab-frequency-input-screening', 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'),
    'IGEM1': ('agent/igem-classification-guidance-screening', 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/IGEM/IGEM1_seed42/weights/best.pt'),
    'SAF1': ('agent/safpn-classification-alignment', 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/SAF1/SAF1_seed42/weights/best.pt'),
    'ACMC1': ('agent/acmc1-residual-error-audit', 'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt'),
}
CHECKPOINTS = {}
for name, (_, relative) in SPECS.items():
    CHECKPOINTS[name] = require_project_artifact(PROJECT_ROOT, relative)
    print(name, '->', CHECKPOINTS[name])
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-multimodel-complementarity-seed42-v1'
EVENT_ROOT = OUTPUT_ROOT / 'events'
EVENT_ROOT.mkdir(parents=True, exist_ok=True)
print('OUTPUT:', OUTPUT_ROOT)


In [ ]:
def checkout_branch(branch):
    subprocess.run(['git', 'fetch', 'origin', branch], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '-B', branch, f'origin/{branch}'], cwd=REPO, check=True)
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    print('BRANCH:', branch, commit)

EVENTS = {}
for name, (branch, _) in SPECS.items():
    event_path = EVENT_ROOT / f'{name}_seed42_events.json'
    EVENTS[name] = event_path
    if event_path.is_file():
        payload = json.loads(event_path.read_text(encoding='utf-8'))
        if payload.get('protocol') == 'faruq-v3-validation-object-events-v1' and payload.get('model') == name and payload.get('seed') == 42:
            print('SKIP existing:', name)
            continue
    checkout_branch(branch)
    command = [sys.executable, str(EXPORTER), '--checkpoint', str(CHECKPOINTS[name]), '--data-root', str(DATA_ROOT), '--output', str(event_path), '--model-name', name, '--seed', '42', '--device', '0']
    subprocess.run(command, cwd=REPO, check=True)
print('SEMUA EVENT SEED42 SELESAI')


In [ ]:
checkout_branch(AUDIT_BRANCH)
event_args = []
for name, path in EVENTS.items():
    event_args += ['--event', f'{name}={path}']
command = [sys.executable, '-m', 'coffee_detector.analysis.multimodel_complementarity_from_events', *event_args, '--output-root', str(OUTPUT_ROOT)]
subprocess.run(command, cwd=REPO, check=True)
SUMMARY = OUTPUT_ROOT / 'multimodel_complementarity_seed42.json'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
print('SUMMARY:', SUMMARY)


In [ ]:
import pandas as pd
from IPython.display import display

model_rows = list(result['models'].values())
display(pd.DataFrame(model_rows).sort_values('accuracy_iou50', ascending=False).style.format({
    'accuracy_iou50': '{:.2%}', 'matched_recall_iou50': '{:.2%}', 'accessibility_iou50': '{:.2%}'
}))

pair_rows = []
for row in result['pairwise_ranked_by_oracle_headroom']:
    pair_rows.append({k: row[k] for k in ('model_a','model_b','a_to_b_rescue_rate','b_to_a_rescue_rate','error_jaccard','oracle_gain_over_best')})
display(pd.DataFrame(pair_rows).style.format({
    'a_to_b_rescue_rate': '{:.2%}', 'b_to_a_rescue_rate': '{:.2%}', 'error_jaccard': '{:.2%}', 'oracle_gain_over_best': '{:.2%}'
}))
print('ALL-MODEL ORACLE:', result['all_model_oracle'])
print('TOP 5 PAIRS + CONFUSION RESCUES:')
for row in result['pairwise_ranked_by_oracle_headroom'][:5]:
    print('\n', row['model_a'], 'vs', row['model_b'])
    print('oracle gain:', f"{row['oracle_gain_over_best']:.2%}", 'Jaccard:', f"{row['error_jaccard']:.2%}")
    print(row['model_a'], 'wrong ->', row['model_b'], 'correct:', row['top_a_wrong_b_rescues'][:8])
    print(row['model_b'], 'wrong ->', row['model_a'], 'correct:', row['top_b_wrong_a_rescues'][:8])
print('Kirim tabel ini untuk interpretasi. Jangan membuka test.')
